# ASTROCARE — Complete Biomedical ML & SHAP Pipeline (Google Colab)

**Disclaimer:** ASTROCARE is an independent research prototype, not a NASA-endorsed product or clinically validated medical diagnostic system. Flagged observations indicate multivariate statistical deviations from a cohort baseline, not medical diagnoses. MMASH is ground-based human physiological data; MMASH, LSDS-7 (CBC), and LSDS-8 (Immune) are strictly separate cohorts and are never merged.

### Pipeline Stages
1. Environment Setup (Free Colab CPU, zero paid APIs)
2. Dataset Upload (`MMASH.zip`, `LSDS-7_Complete_Blood_Count_CBC_TRANSFORMED.csv`, `LSDS-8_Multiplex_serum.immune.EvePanel_SUBMITTED.csv`)
3. ZIP Extraction & File Inventory
4. Dataset Inspection & Supervised Label Eligibility Audit
5. Isolated Cohort Preprocessing (60s MMASH Windowing + HRV, CBC Reference-Range Exclusion + Row 17 Integrity Audit, Immune Per-Analyte Normalization + Censored Values)
6. Unsupervised ML Training (`IsolationForest` + `Modified Z-Score MAD` Baseline)
7. Evaluation (Score Distribution Summary, Contamination Sensitivity, Multi-Seed Stability — no supervised classification metrics without ground truth)
8. Explainable AI (`shap.TreeExplainer` Summary Plots & Per-Observation Drivers)
9. Artifact Export (`models/*.joblib` + `outputs/astrocare_bundle.json`) & Download

## 1. Install Dependencies & Configure Pipeline

In [ ]:
!pip -q install pandas numpy scikit-learn scipy matplotlib seaborn shap joblib

import os
import re
import json
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy.stats import spearmanr
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Configurable audit parameters (synced with TypeScript & FastAPI backend)
MMASH_WINDOW_MODE = "window_60s"      # "window_60s" (60s chronological windows) or "epoch_20"
MMASH_WINDOW_SECONDS = 60
IMMUNE_PROCESSING_MODE = "per_analyte" # "per_analyte", "wide_pivot", or "pooled"
CBC_EXCLUDE_SUSPECT = False            # Set True to exclude suspected corrupted Row 17 during IF fit
CONTAMINATION = 0.05
COLAB_MMASH_REF_WINDOWS = 30377

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/extracted_mmash").mkdir(parents=True, exist_ok=True)
Path("models").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)
print("Environment ready.")

## 2. Dataset Upload
Upload any or all of:
- `MMASH.zip`
- `LSDS-7_Complete_Blood_Count_CBC_TRANSFORMED.csv`
- `LSDS-8_Multiplex_serum.immune.EvePanel_SUBMITTED.csv`

*(If files are already placed in `data/raw/`, skip the interactive prompt cleanly.)*

In [ ]:
existing_raw = [p for p in Path("data/raw").glob("*") if p.is_file()]
if not existing_raw:
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname, content in uploaded.items():
            target = Path("data/raw") / fname
            target.write_bytes(content)
            print(f"Saved {fname} ({len(content):,} bytes) -> {target}")
    except ImportError:
        print("Running outside Google Colab; place raw datasets inside data/raw/ and re-run.")
else:
    print("Found existing files in data/raw:", [p.name for p in existing_raw])

## 3. ZIP Extraction & File Discovery

In [ ]:
raw_files = sorted(Path("data/raw").glob("*"))
zip_files = [p for p in raw_files if p.suffix.lower() == ".zip"]
csv_files = [p for p in raw_files if p.suffix.lower() == ".csv"]

mmash_csvs = []
zip_manifest = []
for zp in zip_files:
    print(f"Extracting ZIP archive: {zp.name}")
    with zipfile.ZipFile(zp, "r") as zf:
        for info in zf.infolist():
            if info.is_dir() or "__MACOSX" in info.filename or Path(info.filename).name.startswith("."):
                continue
            if info.filename.lower().endswith(".csv"):
                zf.extract(info, "data/extracted_mmash")
                extracted_path = Path("data/extracted_mmash") / info.filename
                mmash_csvs.append(extracted_path)
                m = re.search(r"(user[_-]?\d+|subject[_-]?\d+|sub[_-]?\d+|participant[_-]?\d+|s\d{1,3})", info.filename, re.I)
                try:
                    head_df = pd.read_csv(extracted_path, nrows=0)
                    cols = list(head_df.columns)
                    row_cnt = max(0, sum(1 for _ in open(extracted_path, "rb")) - 1)
                except Exception:
                    cols, row_cnt = [], 0
                zip_manifest.append({
                    "path": info.filename,
                    "sizeBytes": int(info.file_size),
                    "rowCount": int(row_cnt),
                    "columns": cols,
                    "inferredSubjectId": m.group(1) if m else None,
                })

cbc_path = next((p for p in csv_files if "cbc" in p.name.lower() or "lsds-7" in p.name.lower()), None)
immune_path = next((p for p in csv_files if "immune" in p.name.lower() or "eve" in p.name.lower() or "lsds-8" in p.name.lower()), None)
phys_csv_path = next((p for p in csv_files if p not in [cbc_path, immune_path]), None)

print(f"MMASH extracted CSVs: {len(mmash_csvs)}")
print(f"Physiological standalone CSV: {phys_csv_path}")
print(f"CBC CSV: {cbc_path}")
print(f"Immune CSV: {immune_path}")

## 4. Dataset Inspection & Provenance Audit

In [ ]:
def inspect_dataframe(df: pd.DataFrame, name: str):
    print("=" * 70)
    print(f"DATASET INSPECTION: {name}")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(f"Columns with missing values ({len(missing)}):")
    if len(missing) > 0:
        print((missing / len(df) * 100).round(2).astype(str) + "%")
    print("\nSample columns:", list(df.columns[:12]))
    print(df.head(3))

if cbc_path:
    cbc_raw_inspect = pd.read_csv(cbc_path)
    inspect_dataframe(cbc_raw_inspect, cbc_path.name)

if immune_path:
    immune_raw_inspect = pd.read_csv(immune_path)
    inspect_dataframe(immune_raw_inspect, immune_path.name)

if mmash_csvs:
    print(f"\nSample MMASH CSVs (showing first 8 of {len(mmash_csvs)}):")
    for p in mmash_csvs[:8]:
        df_sample = pd.read_csv(p, nrows=3)
        print(f" - {p.name} ({p.parent.name}): cols={list(df_sample.columns)}")

## 5 & 6. Isolated Preprocessing, MMASH 60s Windowing, CBC Integrity Audit & Immune Per-Analyte Normalization
Matches the ASTROCARE TypeScript & Python FastAPI logic exactly.

In [ ]:
MISSING_TOKENS = {"", "na", "n/a", "nan", "null", "none", "nil", ".", "-", "nd", "missing"}
DOMAIN_PROVENANCE = {
    "physiological": "MMASH (Multilevel Monitoring of Activity and Sleep in Healthy people) — terrestrial multi-sensor ambulatory physiology benchmark. Not spaceflight telemetry.",
    "cbc": "LSDS-7 Complete Blood Count (CBC) Transformed dataset — hematology measurements evaluated strictly against cohort distributions.",
    "immune": "LSDS-8 Multiplex Serum Immune EvePanel Submitted dataset — cytokine/chemokine multiplex panel analyzed strictly within its own cohort.",
}

def parse_numeric_with_censor(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None, "missing", None
    s = str(val).strip()
    if s.lower() in MISSING_TOKENS:
        return None, "missing", None
    clean = s.replace(",", "")
    if re.match(r"^(?:oor\s*<|blq|lod|<\s*lod|<\s*lloq|<\s*dl|below\s*(?:lod|dl|range))$", clean, re.I):
        return None, "left", None
    if re.match(r"^(?:oor\s*>|>\s*uloq|above\s*(?:uloq|range)|alr)$", clean, re.I):
        return None, "right", None
    m_left = re.match(r"^(?:oor\s*)?<=?\s*(-?\d+(?:\.\d+)?(?:e[+-]?\d+)?)$", clean, re.I)
    if m_left:
        b = float(m_left.group(1))
        return round(b * 0.5, 6), "left", b
    m_right = re.match(r"^(?:oor\s*)?>=?\s*(-?\d+(?:\.\d+)?(?:e[+-]?\d+)?)$", clean, re.I)
    if m_right:
        b = float(m_right.group(1))
        return b, "right", b
    stripped = re.sub(r"\*+$", "", clean)
    try:
        num = float(stripped)
        if np.isfinite(num):
            return num, "none", None
    except ValueError:
        pass
    return None, "missing", None

def _compute_hrv_stats(ibis):
    valid_ms = [v * 1000.0 if v < 10.0 else v for v in ibis]
    valid_ms = np.array([ms for ms in valid_ms if 250.0 <= ms <= 2500.0], dtype=float)
    if len(valid_ms) < 2:
        return None
    mean_ms = float(np.mean(valid_ms))
    hr_mean = 60000.0 / mean_ms
    sdnn = float(np.std(valid_ms, ddof=1))
    diffs = np.diff(valid_ms)
    rmssd = float(np.sqrt(np.mean(diffs ** 2)))
    return round(hr_mean, 3), round(sdnn, 3), round(rmssd, 3)

def _parse_time_seconds(raw):
    s = str(raw).strip()
    m = re.match(r"^(\d{1,2}):(\d{2})(?::(\d{2})(?:\.\d+)?)?$", s)
    if m:
        hh, mm = int(m.group(1)), int(m.group(2))
        ss = float(m.group(3)) if m.group(3) else 0.0
        if 0 <= hh < 48 and 0 <= mm < 60 and 0 <= ss < 60:
            return hh * 3600.0 + mm * 60.0 + ss, False
    try:
        dt = pd.to_datetime(s, errors="raise", utc=True)
        return float(dt.timestamp()), True
    except Exception:
        return None, False

def _format_bucket_label(bucket_sec: float, is_epoch: bool) -> str:
    if is_epoch:
        return datetime.fromtimestamp(bucket_sec, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    day = int(bucket_sec // 86400) + 1
    rem = max(0, int(bucket_sec % 86400))
    hh, mm, ss = rem // 3600, (rem % 3600) // 60, rem % 60
    return f"D{day} {hh:02d}:{mm:02d}:{ss:02d}"

def build_mmash_cohort_table(csv_paths, fallback_csv=None, mode="window_60s", window_seconds=60):
    if fallback_csv is not None:
        return pd.read_csv(fallback_csv, na_values=list(MISSING_TOKENS)), fallback_csv.name
    if not csv_paths:
        return None, None

    by_subject = {}
    for p in csv_paths:
        fname_l = p.name.lower()
        if "user_info" in fname_l or "questionnaire" in fname_l:
            continue
        sub_df = pd.read_csv(p, na_values=list(MISSING_TOKENS))
        if sub_df.empty:
            continue
        m = re.search(r"(user[_-]?\d+|subject[_-]?\d+|sub[_-]?\d+|participant[_-]?\d+|s\d{1,3})", str(p), re.I)
        subj = m.group(1) if m else p.parent.name
        base = p.stem
        s_entry = by_subject.setdefault(subj, {"epochs": {}, "summary": {}})

        num_cols = [c for c in sub_df.columns if not re.match(r"^(day|unnamed|index)$", str(c).strip(), re.I) and pd.api.types.is_numeric_dtype(sub_df[c])]
        if len(sub_df) <= 25:
            for nc in num_cols:
                vals = sub_df[nc].dropna()
                if len(vals) > 0:
                    s_entry["summary"][f"{base}_{nc}"] = round(float(vals.mean()), 4)
            continue

        time_col = next((c for c in sub_df.columns if re.match(r"^(time|timestamp|datetime|date_time|start)$", str(c).strip(), re.I)), None)
        day_col = next((c for c in sub_df.columns if re.match(r"^(day|day_index)$", str(c).strip(), re.I)), None)
        ibi_col = next((c for c in num_cols if re.search(r"ibi|rr", str(c), re.I)), None)

        if mode == "window_60s":
            buckets = {}
            day_rollover = 0
            prev_hms = None
            cum_ibi = 0.0
            is_epoch_ts = False
            records = sub_df.to_dict(orient="records")
            for idx_r, r in enumerate(records):
                bkey = None
                if time_col and pd.notna(r.get(time_col)):
                    t_sec, is_ep = _parse_time_seconds(r[time_col])
                    if t_sec is not None:
                        if is_ep:
                            is_epoch_ts = True
                            bkey = int(t_sec // window_seconds)
                        else:
                            day_off = 0
                            if day_col and pd.notna(r.get(day_col)):
                                try:
                                    dv = float(r[day_col])
                                    if dv >= 1: day_off = int(dv) - 1
                                except Exception: pass
                            else:
                                if prev_hms is not None and t_sec < prev_hms - 43200:
                                    day_rollover += 1
                                day_off = day_rollover
                                prev_hms = t_sec
                            bkey = int((day_off * 86400.0 + t_sec) // window_seconds)
                if bkey is None:
                    if ibi_col and pd.notna(r.get(ibi_col)):
                        iv = float(r[ibi_col])
                        ibi_s = iv / 1000.0 if iv > 10.0 else iv
                        bkey = int(cum_ibi // window_seconds)
                        cum_ibi += ibi_s
                    else:
                        bkey = idx_r // window_seconds
                buckets.setdefault(bkey, []).append(r)

            for bkey in sorted(buckets.keys()):
                sl_df = pd.DataFrame(buckets[bkey])
                ep = s_entry["epochs"].setdefault(bkey, {"subject_id": subj})
                ep[f"{base}_time"] = _format_bucket_label(bkey * window_seconds, is_epoch_ts)
                for nc in num_cols:
                    vals = pd.to_numeric(sl_df[nc], errors="coerce").dropna()
                    if len(vals) > 0:
                        ep[f"{base}_{nc}_mean"] = round(float(vals.mean()), 4)
                if ibi_col:
                    ibis = pd.to_numeric(sl_df[ibi_col], errors="coerce").dropna().tolist()
                    hrv = _compute_hrv_stats(ibis)
                    if hrv:
                        ep[f"{base}_HR_bpm_mean"], ep[f"{base}_SDNN_ms"], ep[f"{base}_RMSSD_ms"] = hrv
        else:
            n_ep = min(20, len(sub_df))
            step = int(np.ceil(len(sub_df) / n_ep))
            for w in range(n_ep):
                sl_df = sub_df.iloc[w * step : (w + 1) * step]
                if sl_df.empty: continue
                bkey = w + 1
                ep = s_entry["epochs"].setdefault(bkey, {"subject_id": subj})
                if time_col and pd.notna(sl_df[time_col].iloc[0]):
                    ep[f"{base}_time"] = str(sl_df[time_col].iloc[0])
                for nc in num_cols:
                    vals = pd.to_numeric(sl_df[nc], errors="coerce").dropna()
                    if len(vals) > 0:
                        ep[f"{base}_{nc}_mean"] = round(float(vals.mean()), 4)
                if ibi_col:
                    ibis = pd.to_numeric(sl_df[ibi_col], errors="coerce").dropna().tolist()
                    hrv = _compute_hrv_stats(ibis)
                    if hrv:
                        ep[f"{base}_HR_bpm_mean"], ep[f"{base}_SDNN_ms"], ep[f"{base}_RMSSD_ms"] = hrv

    def _nat_key(s):
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

    aligned = []
    for subj in sorted(by_subject.keys(), key=_nat_key):
        data = by_subject[subj]
        if data["epochs"]:
            for idx_ep, bkey in enumerate(sorted(data["epochs"].keys()), start=1):
                merged = {**data["epochs"][bkey], **data["summary"], "epoch_index": idx_ep}
                tkey = next((k for k in merged if k.endswith("_time") and merged[k]), None)
                if tkey and "timestamp" not in merged:
                    merged["timestamp"] = merged[tkey]
                aligned.append(merged)
        elif data["summary"]:
            aligned.append({"subject_id": subj, **data["summary"]})

    fname = zip_files[0].name if zip_files else "MMASH.zip"
    return pd.DataFrame(aligned), fname

def audit_cbc_consistency(df_raw: pd.DataFrame, subject_ids):
    cols = list(df_raw.columns)
    find_col = lambda pat: next((c for c in cols if re.search(pat, c, re.I) and not re.search(r"range", c, re.I)), None)
    wbc_c = find_col(r"white_blood_cell|wbc")
    plt_c = find_col(r"platelet_count")
    mpv_c = find_col(r"(?:^|_)mpv(?:_|$)")
    rdw_c = find_col(r"(?:^|_)rdw(?:_|$)")
    aneut_c = find_col(r"absolute_neutrophils")
    alym_c = find_col(r"absolute_lymphocytes")
    pneut_c = find_col(r"neutrophils_value_percent")
    plym_c = find_col(r"lymphocytes_value_percent")
    findings, suspect_idx = [], set()
    for i in range(len(df_raw)):
        row_f, rec, exp = [], {}, {}
        getv = lambda c: float(df_raw[c].iloc[i]) if c and pd.notna(df_raw[c].iloc[i]) else None
        plt_v, mpv_v, rdw_v, wbc_v = getv(plt_c), getv(mpv_c), getv(rdw_c), getv(wbc_c)
        an_v, al_v, pn_v, pl_v = getv(aneut_c), getv(alym_c), getv(pneut_c), getv(plym_c)
        if plt_v is not None and plt_v < 50:
            rec["platelet_count"] = plt_v
            row_f.append(f"Platelet count {plt_v} x10^3/uL is implausibly low.")
        if mpv_v is not None and mpv_v > 30:
            rec["mpv"] = mpv_v
            row_f.append(f"MPV {mpv_v} fL is physiologically impossible (matches platelet scale ~330).")
        if rdw_v is not None and rdw_v > 25:
            rec["rdw"] = rdw_v
            row_f.append(f"RDW {rdw_v}% is >2.5x cohort baseline.")
        if wbc_v is not None and pn_v is not None and an_v is not None:
            exp_n = round(wbc_v * 1000.0 * (pn_v / 100.0), 1)
            exp["expected_abs_neutrophils"] = exp_n
            if abs(an_v - exp_n) > max(500.0, exp_n * 0.25):
                rec["absolute_neutrophils"] = an_v
                row_f.append(f"WBC x Neut% predicts {exp_n} cells/uL, recorded {an_v}.")
        if wbc_v is not None and pl_v is not None and al_v is not None:
            exp_l = round(wbc_v * 1000.0 * (pl_v / 100.0), 1)
            exp["expected_abs_lymphocytes"] = exp_l
            if abs(al_v - exp_l) > max(500.0, exp_l * 0.25):
                rec["absolute_lymphocytes"] = al_v
                row_f.append(f"WBC x Lymph% predicts {exp_l} cells/uL, recorded {al_v}.")
        if row_f:
            sev = "critical" if len(row_f) >= 2 else "warning"
            if sev == "critical":
                suspect_idx.add(i)
            findings.append({
                "rowIndex": i + 1,
                "sampleName": str(subject_ids[i]),
                "severity": sev,
                "findings": row_f,
                "recordedValues": rec,
                "expectedFromWbcDiff": exp,
                "handling": "Retained unmodified in raw source. Available as verified-only exclusion toggle during Isolation Forest fitting.",
            })
    return findings, suspect_idx

def preprocess_dataframe(df: pd.DataFrame, domain: str, filename: str, cbc_exclude_suspect=False, immune_mode="per_analyte", window_mode="window_60s"):
    raw_rows = len(df)
    df_clean = df.drop_duplicates().reset_index(drop=True)
    cols = list(df_clean.columns)

    subject_col = next((c for c in cols if re.match(r"^(subject|participant|user|astronaut|crew|animal|mouse|patient|sample|specimen)[_\s-]*(id|num|code|name)?$", c, re.I) or c.lower() in {"subject_id", "id", "sample name"}), None)
    timestamp_col = next((c for c in cols if c != subject_col and re.search(r"timestamp|date|time|collection|day|epoch", c, re.I) and not pd.api.types.is_numeric_dtype(df_clean[c])), None)
    label_candidates = [c for c in cols if re.match(r"^(label|target|y|outcome|diagnosis|anomaly|is_anomaly|ground_truth|class)$", c, re.I)]

    subject_ids = df_clean[subject_col].fillna("unknown").astype(str).tolist() if subject_col else [f"row_{i+1}" for i in range(len(df_clean))]
    timestamps = [None if pd.isna(v) else str(v) for v in df_clean[timestamp_col].tolist()] if timestamp_col else [None] * len(df_clean)

    cbc_findings, suspect_indices = ([], set())
    if domain == "cbc":
        cbc_findings, suspect_indices = audit_cbc_consistency(df_clean, subject_ids)
        if cbc_exclude_suspect and suspect_indices:
            keep_mask = [i not in suspect_indices for i in range(len(df_clean))]
            df_clean = df_clean.loc[keep_mask].reset_index(drop=True)
            subject_ids = [s for i, s in enumerate(subject_ids) if i not in suspect_indices]
            timestamps = [t for i, t in enumerate(timestamps) if i not in suspect_indices]

    integrity_statuses = ["suspected_corrupted" if (domain == "cbc" and not cbc_exclude_suspect and i in suspect_indices) else "verified" for i in range(len(df_clean))]

    # Immune censored parsing & per-analyte normalization
    analyte_labels = None
    immune_audit = None
    if domain == "immune":
        analyte_col = next((c for c in cols if re.match(r"^(analyte|cytokine|biomarker|marker|assay|target_analyte)$", c, re.I)), None)
        conc_col = next((c for c in cols if re.match(r"^(concentration|conc|value|observed_concentration|result)$", c, re.I)), None)
        pct_col = next((c for c in cols if re.match(r"^(percent|pct|pct_change)$", c, re.I)), None)
        left_c, right_c, miss_c = 0, 0, 0
        if conc_col:
            parsed_vals, cens_flags = [], []
            for v in df_clean[conc_col].tolist():
                nv, cf, _ = parse_numeric_with_censor(v)
                if cf == "left": left_c += 1
                elif cf == "right": right_c += 1
                elif cf == "missing": miss_c += 1
                parsed_vals.append(nv)
                cens_flags.append(cf)
            df_clean[conc_col] = pd.Series(parsed_vals, dtype=float)
        if pct_col:
            df_clean[pct_col] = pd.Series([parse_numeric_with_censor(v)[0] for v in df_clean[pct_col].tolist()], dtype=float)
        if analyte_col:
            analyte_labels = df_clean[analyte_col].fillna("Unknown").astype(str).tolist()
            if immune_mode == "per_analyte" and conc_col:
                df_clean["Concentration_analyte_z"] = df_clean.groupby(analyte_col)[conc_col].transform(
                    lambda s: (s.fillna(s.median()) - s.fillna(s.median()).mean()) / (s.fillna(s.median()).std(ddof=1) or 1.0)
                )
                if pct_col:
                    df_clean["Percent_analyte_z"] = df_clean.groupby(analyte_col)[pct_col].transform(
                        lambda s: (s.fillna(s.median()) - s.fillna(s.median()).mean()) / (s.fillna(s.median()).std(ddof=1) or 1.0)
                    )
            immune_audit = {
                "activeMode": immune_mode,
                "hasAnalyteColumn": True,
                "analyteCount": int(len(set(analyte_labels))),
                "censoredValuesSummary": {
                    "leftCensoredCount": left_c, "rightCensoredCount": right_c, "missingCount": miss_c,
                    "handlingRule": "Left-censored (<x) -> 0.5x bound; right-censored (>x) -> bound; remaining missing values median-imputed per analyte."
                },
                "perAnalyteSummary": [],
                "notes": [f"Preserved {len(set(analyte_labels))} unique analytes; applied {immune_mode} standardization."]
            }
        else:
            immune_audit = {
                "activeMode": "pooled", "hasAnalyteColumn": False, "analyteCount": 0,
                "censoredValuesSummary": {"leftCensoredCount": left_c, "rightCensoredCount": right_c, "missingCount": miss_c, "handlingRule": "Standard median imputation."},
                "perAnalyteSummary": [], "notes": ["No Analyte column detected; using standard numeric preprocessing."]
            }

    excluded_ref, excluded_non_meas, dropped_high_missing, excluded_zero_var = [], [], [], []
    feature_cols, col_profiles = [], []
    is_ref_col = lambda n: bool(re.search(r"(?:^|_)range_(?:min|max)(?:_|$)|(?:^|_)(?:ref|reference)_(?:min|max|low|high)(?:_|$)", n, re.I))
    is_non_meas = lambda n: bool(re.match(r"^(epoch_index|row_index|unnamed:\s*\d+|index)$", n, re.I) or (domain == "physiological" and re.search(r"_(?:in bed date|out bed date|onset date|day)$", n, re.I)))

    for c in list(df_clean.columns):
        coerced = pd.to_numeric(df_clean[c].astype(str).str.replace(",", "", regex=False), errors="coerce")
        non_null_raw = int(df_clean[c].notna().sum())
        is_num = pd.api.types.is_numeric_dtype(df_clean[c]) or (non_null_raw > 0 and coerced.notna().sum() / max(1, non_null_raw) >= 0.7)
        missing_count = int(coerced.isna().sum()) if is_num else int(df_clean[c].isna().sum())
        missing_pct = round(missing_count / max(1, len(df_clean)) * 100.0, 2)

        role, reason = "ignored", None
        if c == subject_col: role, reason = "subject_id", "Subject/sample identifier"
        elif c == timestamp_col: role, reason = "timestamp", "Timestamp / timepoint metadata"
        elif c in label_candidates: role, reason = "label_candidate", "Target label candidate"
        elif is_ref_col(c):
            role, reason = "ignored", "Clinical reference-range bound (excluded from ML)"
            excluded_ref.append(c)
        elif is_non_meas(c):
            role, reason = "ignored", "Ordinal epoch/date index (non-physiological metadata)"
            excluded_non_meas.append(c)
        elif domain == "immune" and immune_mode == "per_analyte" and immune_audit and immune_audit["hasAnalyteColumn"] and c in {"Concentration", "Percent"}:
            role, reason = "ignored", "Raw multi-scale value replaced by per-analyte z-score feature"
        elif is_num:
            role = "feature"

        stats = {"min": None, "max": None, "mean": None, "median": None, "std": None, "q25": None, "q75": None, "iqrOutlierCount": 0}
        if is_num:
            valid = coerced.dropna()
            if len(valid) > 0:
                q25, q75 = float(valid.quantile(0.25)), float(valid.quantile(0.75))
                iqr = q75 - q25
                outliers = int(((valid < q25 - 1.5 * iqr) | (valid > q75 + 1.5 * iqr)).sum()) if iqr > 0 else 0
                stats = {
                    "min": round(float(valid.min()), 4), "max": round(float(valid.max()), 4),
                    "mean": round(float(valid.mean()), 4), "median": round(float(valid.median()), 4),
                    "std": round(float(valid.std(ddof=1)) if len(valid) > 1 else 0.0, 4),
                    "q25": round(q25, 4), "q75": round(q75, 4), "iqrOutlierCount": outliers,
                }
            if role == "feature":
                if missing_pct > 50.0:
                    role, reason = "ignored", "Dropped (>50% missingness)"
                    dropped_high_missing.append(c)
                elif stats["std"] is None or stats["std"] <= 1e-12:
                    role, reason = "ignored", "Zero variance (constant across cohort)"
                    excluded_zero_var.append(c)
                else:
                    df_clean[c] = coerced
                    feature_cols.append(c)

        col_profiles.append({
            "name": c, "inferredType": "numeric" if is_num else "categorical", "role": role,
            "unit": None, "nonNullCount": int(len(df_clean) - missing_count),
            "missingCount": missing_count, "missingPct": missing_pct,
            "uniqueCount": int(df_clean[c].nunique(dropna=True)), "excludedReason": reason, **stats
        })

    engineered = []
    nlr_audit = None
    if domain == "cbc":
        neut = next((c for c in feature_cols if re.search(r"neutrophil|neut", c, re.I) and not re.search(r"pct|percent|%", c, re.I)), None)
        lymph = next((c for c in feature_cols if re.search(r"lymphocyte|lymph", c, re.I) and not re.search(r"pct|percent|%", c, re.I)), None)
        if neut and lymph:
            nlr = df_clean[neut] / df_clean[lymph].replace(0, np.nan)
            if nlr.notna().sum() >= max(3, int(0.5 * len(df_clean))):
                df_clean["NLR_engineered"] = nlr
                feature_cols.append("NLR_engineered")
                engineered.append("NLR_engineered")
                valid_nlr = nlr.dropna()
                nlr_audit = {
                    "computed": True, "numeratorColumn": neut, "denominatorColumn": lymph,
                    "unitCheck": "Both columns share matching cells/uL units.",
                    "zeroOrNearZeroDenominators": int((df_clean[lymph] <= 0).sum()),
                    "missingNlrCount": int(nlr.isna().sum()),
                    "minNlr": round(float(valid_nlr.min()), 4), "maxNlr": round(float(valid_nlr.max()), 4),
                    "medianNlr": round(float(valid_nlr.median()), 4), "pctCrossCheckDiscrepancies": [],
                    "notes": [f"Computed NLR_engineered across {len(valid_nlr)} rows (median {valid_nlr.median():.3f})."]
                }

    missing_before = int(df_clean[feature_cols].isna().sum().sum())
    imputer = SimpleImputer(strategy="median")
    X_imp = imputer.fit_transform(df_clean[feature_cols])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)

    mmash_cmp = None
    if domain == "physiological":
        mmash_cmp = {
            "activeMode": window_mode, "windowSeconds": MMASH_WINDOW_SECONDS,
            "rawZipAvailable": len(mmash_csvs) > 0,
            "colabReferenceWindows": COLAB_MMASH_REF_WINDOWS,
            "activeRowCount": int(len(df_clean)), "activeSubjectCount": int(len(set(subject_ids))),
            "window60sRowCount": int(len(df_clean)) if window_mode == "window_60s" else None,
            "epoch20RowCount": int(len(df_clean)) if window_mode == "epoch_20" else None,
            "deltaVsColabWindows": int(len(df_clean) - COLAB_MMASH_REF_WINDOWS) if window_mode == "window_60s" else None,
            "validationStatus": "validated_against_zip" if len(mmash_csvs) > 0 else "epoch_20_overview",
            "featureDistributionComparisons": [],
            "notes": [f"Colab produced {len(df_clean):,} {window_mode} windows across {len(set(subject_ids))} subjects (Colab benchmark: {COLAB_MMASH_REF_WINDOWS:,})."]
        }

    metadata = {
        "id": f"{domain}_colab", "domain": domain, "filename": filename, "fileType": "colab_bundle",
        "provenanceNote": DOMAIN_PROVENANCE[domain],
        "uploadedAt": datetime.now(timezone.utc).isoformat(),
        "rowCount": int(len(df_clean)), "rawRowCount": int(raw_rows),
        "duplicateCount": int(raw_rows - len(df_clean)), "columnCount": int(len(cols)),
        "subjectCount": int(len(set(subject_ids))), "subjectIds": list(dict.fromkeys(subject_ids))[:100],
        "subjectIdColumn": subject_col, "hasTimestamps": any(t is not None for t in timestamps),
        "timestampColumn": timestamp_col, "timestampRange": None, "columns": col_profiles,
        "droppedHighMissingColumns": dropped_high_missing,
        "imputedCellCount": missing_before,
        "engineeredFeatures": engineered,
        "supervisedLabelAudit": {
            "hasValidLabels": len(label_candidates) > 0,
            "candidateColumns": label_candidates,
            "reason": "No ground-truth anomaly/diagnostic labels exist; supervised accuracy/precision/recall/F1 are intentionally omitted."
        },
        "qualityNotes": [f"Preprocessed {len(df_clean):,} rows and {len(feature_cols)} features in Colab."],
        "dataQualityAudit": {
            "missingCellsBeforeInFeatures": missing_before, "missingCellsBeforeTotal": missing_before,
            "missingCellsAfter": 0,
            "excludedReferenceRangeColumns": excluded_ref,
            "excludedNonMeasurementColumns": excluded_non_meas,
            "excludedHighMissingColumns": dropped_high_missing,
            "excludedZeroVarianceColumns": excluded_zero_var,
            "cbcConsistencyFindings": cbc_findings,
            "cbcIntegrityComparison": None,
            "nlrAudit": nlr_audit,
            "mmashWindowComparison": mmash_cmp,
            "immuneAudit": immune_audit,
            "windowingNote": f"Processed with {window_mode}." if domain == "physiological" else None,
            "cohortIsolationNote": "Strictly isolated cohort — never concatenated with unrelated studies."
        },
        "zipManifest": zip_manifest if (domain == "physiological" and zip_manifest) else None,
    }
    return {
        "metadata": metadata, "feature_cols": feature_cols, "X_imp": X_imp, "X_scaled": X_scaled,
        "scaler": scaler, "imputer": imputer, "subject_ids": subject_ids, "timestamps": timestamps,
        "analyte_labels": analyte_labels, "integrity_statuses": integrity_statuses
    }

cohorts = {}
phys_df, phys_name = build_mmash_cohort_table(mmash_csvs, phys_csv_path, mode=MMASH_WINDOW_MODE, window_seconds=MMASH_WINDOW_SECONDS)
if phys_df is not None and not phys_df.empty:
    cohorts["physiological"] = preprocess_dataframe(phys_df, "physiological", phys_name, window_mode=MMASH_WINDOW_MODE)
if cbc_path:
    cbc_df_raw = pd.read_csv(cbc_path, na_values=list(MISSING_TOKENS))
    cohorts["cbc"] = preprocess_dataframe(cbc_df_raw, "cbc", cbc_path.name, cbc_exclude_suspect=CBC_EXCLUDE_SUSPECT)
if immune_path:
    imm_df_raw = pd.read_csv(immune_path, na_values=list(MISSING_TOKENS))
    cohorts["immune"] = preprocess_dataframe(imm_df_raw, "immune", immune_path.name, immune_mode=IMMUNE_PROCESSING_MODE)

for d, c in cohorts.items():
    print(f"[{d}] rows={c['metadata']['rowCount']:,}, features={len(c['feature_cols'])}, missing_before={c['metadata']['imputedCellCount']}")

## 7, 8 & 9. Unsupervised Training, Sensitivity & Stability Audits, and SHAP Explainable AI

In [ ]:
bundle_export = {"datasets": {}, "analyses": {}}
MODEL_FILENAMES = {
    "physiological": "physiological_anomaly_model.joblib",
    "cbc": "cbc_anomaly_model.joblib",
    "immune": "immune_anomaly_model.joblib",
}

for domain, c in cohorts.items():
    print("\n" + "=" * 70)
    print(f"TRAINING & SHAP EXPLAINABILITY: {domain.upper()}")
    X_scaled, X_imp = c["X_scaled"], c["X_imp"]
    feats = c["feature_cols"]
    n = len(X_scaled)
    max_samples = min(256, n)

    model = IsolationForest(n_estimators=150, max_samples=max_samples, contamination=CONTAMINATION, random_state=42)
    model.fit(X_scaled)
    scores = -model.score_samples(X_scaled)
    dec_scores = model.decision_function(X_scaled)
    is_anom = model.predict(X_scaled) == -1
    threshold_score = float(scores[is_anom].min()) if is_anom.any() else float(scores.max())

    # Modified Z-Score (MAD) Baseline
    medians = np.median(X_imp, axis=0)
    mads = np.median(np.abs(X_imp - medians), axis=0)
    stds = np.where(X_imp.std(axis=0, ddof=1) > 1e-12, X_imp.std(axis=0, ddof=1), 1.0)
    mads = np.where(mads > 1e-12, mads, stds * 0.6745)
    mod_z = np.abs(0.6745 * (X_imp - medians) / mads)
    base_max_z = mod_z.max(axis=1)
    base_flags = base_max_z > 3.5

    # SHAP TreeExplainer (all rows for n <= 2500; all flagged + deterministic sample up to 2,000 for very large MMASH)
    explainer = shap.TreeExplainer(model)
    if n <= 2500:
        shap_indices = np.arange(n)
    else:
        rng = np.random.default_rng(42)
        anom_idx = np.where(is_anom)[0]
        norm_idx = np.where(~is_anom)[0]
        sample_norm = rng.choice(norm_idx, size=min(1500, len(norm_idx)), replace=False)
        shap_indices = np.sort(np.unique(np.concatenate([anom_idx, sample_norm])))

    shap_sub = -np.asarray(explainer.shap_values(X_scaled[shap_indices]), dtype=float)
    shap_lookup = {int(idx): shap_sub[k] for k, idx in enumerate(shap_indices)}
    abs_shap_sub = np.abs(shap_sub)
    sub_anom_mask = is_anom[shap_indices]
    global_imp = abs_shap_sub[sub_anom_mask].mean(axis=0) if sub_anom_mask.any() else abs_shap_sub.mean(axis=0)
    global_imp = global_imp / (global_imp.sum() or 1.0)

    plt.figure(figsize=(8, 4.5))
    shap.summary_plot(shap_sub, pd.DataFrame(X_scaled[shap_indices], columns=feats), show=False)
    plt.title(f"ASTROCARE SHAP Summary — {domain.upper()} (n_explained={len(shap_indices):,})")
    plt.tight_layout()
    plt.savefig(f"outputs/shap_summary_{domain}.png", dpi=150)
    plt.show()

    # Contamination sensitivity
    sensitivity = []
    for cont in [0.01, 0.03, 0.05, 0.08, 0.10, 0.15]:
        k_c = max(1, int(round(n * cont)))
        top_idx = set(np.argsort(scores)[::-1][:k_c].tolist())
        thr_c = float(np.sort(scores)[::-1][k_c - 1])
        base_idx = set(np.where(base_flags)[0].tolist())
        overlap = (len(top_idx & base_idx) / max(1, len(top_idx))) * 100.0
        sensitivity.append({
            "contamination": cont, "flaggedCount": k_c, "flaggedPct": round(k_c / n * 100.0, 2),
            "thresholdScore": round(thr_c, 6), "baselineOverlapPct": round(overlap, 2)
        })

    # Stability across 5 random seeds
    seeds = [42, 101, 202, 303, 404]
    seed_scores = [scores]
    k_top = max(1, int(round(n * CONTAMINATION)))
    seed_sets = [set(np.argsort(scores)[::-1][:k_top].tolist())]
    for s in seeds[1:]:
        ms = IsolationForest(n_estimators=100, max_samples=max_samples, contamination=CONTAMINATION, random_state=s).fit(X_scaled)
        sc = -ms.score_samples(X_scaled)
        seed_scores.append(sc)
        seed_sets.append(set(np.argsort(sc)[::-1][:k_top].tolist()))
    jaccards = [len(seed_sets[i] & seed_sets[j]) / max(1, len(seed_sets[i] | seed_sets[j])) for i in range(5) for j in range(i + 1, 5)]
    spearmans = [float(spearmanr(seed_scores[i], seed_scores[j])[0]) for i in range(5) for j in range(i + 1, 5)]

    # Immune per-analyte anomaly breakdown
    if domain == "immune" and c["metadata"]["dataQualityAudit"]["immuneAudit"] and c["analyte_labels"]:
        ia = c["metadata"]["dataQualityAudit"]["immuneAudit"]
        per_an = []
        an_arr = np.array(c["analyte_labels"])
        for an_name in sorted(set(c["analyte_labels"])):
            mask = an_arr == an_name
            rc = int(mask.sum())
            ifc = int((is_anom & mask).sum())
            madc = int((base_flags & mask).sum())
            per_an.append({
                "analyte": an_name, "unit": None, "rowCount": rc, "missingCount": 0,
                "leftCensoredCount": 0, "rightCensoredCount": 0, "median": None, "iqr": None,
                "ifAnomalyCount": ifc, "madFlagCount": madc, "anomalyPct": round(ifc / max(1, rc) * 100.0, 2)
            })
        ia["perAnalyteSummary"] = sorted(per_an, key=lambda x: (-x["ifAnomalyCount"], -x["madFlagCount"]))

    # Save joblib model
    artifact = {
        "domain": domain, "model": model, "scaler": c["scaler"], "imputer": c["imputer"],
        "feature_columns": feats, "contamination": CONTAMINATION, "trained_at": c["metadata"]["uploadedAt"]
    }
    joblib.dump(artifact, f"models/{MODEL_FILENAMES[domain]}")
    print(f"Saved model -> models/{MODEL_FILENAMES[domain]} (Jaccard={np.mean(jaccards):.4f}, Spearman={np.mean(spearmans):.4f})")

    # Build ASTROCARE Web UI bundle
    means = X_imp.mean(axis=0)
    records = []
    keep_top_n = len(feats) if n <= 3000 else 6
    for i in range(n):
        sv = shap_lookup.get(i)
        abs_sv = np.abs(sv) if sv is not None else np.abs(X_scaled[i])
        row_sum = float(abs_sv.sum()) or 1.0
        contribs = sorted([
            {
                "feature": feats[j],
                "value": round(float(X_imp[i, j]), 4),
                "cohortMean": round(float(means[j]), 4),
                "cohortStd": round(float(stds[j]), 4),
                "zScore": round(float(X_scaled[i, j]), 3),
                "isolationContribution": round(float(sv[j] if sv is not None else abs_sv[j]), 4),
                "contributionPct": round(float(abs_sv[j] / row_sum * 100.0), 1),
                "direction": "elevated" if X_scaled[i, j] > 1.0 else ("depressed" if X_scaled[i, j] < -1.0 else "nominal")
            }
            for j in range(len(feats))
        ], key=lambda x: x["contributionPct"], reverse=True)
        if not is_anom[i] and n > 3000:
            contribs = contribs[:keep_top_n]
        records.append({
            "index": i + 1, "subjectId": c["subject_ids"][i], "timestamp": c["timestamps"][i],
            "analyte": c["analyte_labels"][i] if c["analyte_labels"] else None,
            "integrityStatus": c["integrity_statuses"][i],
            "values": {feats[j]: round(float(X_imp[i, j]), 4) for j in range(len(feats))},
            "anomalyScore": round(float(scores[i]), 6), "decisionScore": round(float(dec_scores[i]), 6),
            "isAnomaly": bool(is_anom[i]), "baselineMaxModifiedZ": round(float(base_max_z[i]), 4),
            "baselineFlag": bool(base_flags[i]), "featureContributions": contribs,
            "explanation": f"Observation #{i+1} ({c['subject_ids'][i]}) scored {scores[i]:.3f} via Colab IsolationForest + SHAP."
        })

    counts, edges = np.histogram(scores, bins=12)
    hist = [{
        "binLabel": f"{edges[b]:.2f}", "binStart": round(float(edges[b]), 4), "binEnd": round(float(edges[b + 1]), 4),
        "normalCount": int(((scores >= edges[b]) & (scores <= edges[b + 1] if b == 11 else scores < edges[b + 1]) & ~is_anom).sum()),
        "anomalyCount": int(((scores >= edges[b]) & (scores <= edges[b + 1] if b == 11 else scores < edges[b + 1]) & is_anom).sum())
    } for b in range(len(counts))]

    corr_feats = feats[:14]
    corr_df = pd.DataFrame(X_imp, columns=feats)[corr_feats].corr().fillna(0.0)

    bundle_export["datasets"][domain] = c["metadata"]
    bundle_export["analyses"][domain] = {
        "domain": domain, "trained": True, "source": "colab_python_sklearn",
        "modelType": "sklearn.ensemble.IsolationForest (n_estimators=150)",
        "baselineType": "Modified Z-Score (MAD > 3.5)",
        "modelParams": {
            "algorithm": "sklearn.ensemble.IsolationForest", "nEstimators": 150,
            "maxSamples": int(max_samples), "maxDepth": int(np.ceil(np.log2(max(2, max_samples)))),
            "contamination": CONTAMINATION, "randomSeed": 42,
            "scaler": "StandardScaler", "imputer": "SimpleImputer(median)"
        },
        "scoreSummary": {
            "min": round(float(np.min(scores)), 4), "q25": round(float(np.quantile(scores, 0.25)), 4),
            "median": round(float(np.median(scores)), 4), "mean": round(float(np.mean(scores)), 4),
            "q75": round(float(np.quantile(scores, 0.75)), 4), "p95": round(float(np.quantile(scores, 0.95)), 4),
            "max": round(float(np.max(scores)), 4)
        },
        "trainedAt": datetime.now(timezone.utc).isoformat(),
        "sampleCount": n, "featureColumns": feats, "contamination": CONTAMINATION,
        "thresholdScore": round(threshold_score, 6), "anomalyCount": int(is_anom.sum()),
        "normalCount": int((~is_anom).sum()), "anomalyPercentage": round(float(is_anom.mean() * 100.0), 2),
        "scoreDistribution": hist,
        "featureImportance": sorted([{"feature": feats[j], "importance": round(float(global_imp[j]), 4), "meanAnomalyAbsZ": round(float(np.abs(X_scaled[is_anom, j]).mean()) if is_anom.any() else 0.0, 3), "meanNormalAbsZ": round(float(np.abs(X_scaled[~is_anom, j]).mean()) if (~is_anom).any() else 0.0, 3), "method": "SHAP"} for j in range(len(feats))], key=lambda x: x["importance"], reverse=True),
        "contaminationSensitivity": sensitivity,
        "stabilityAnalysis": {"numSeeds": 5, "meanJaccardOverlap": round(float(np.mean(jaccards)), 4), "meanSpearmanRankCorr": round(float(np.mean(spearmans)), 4), "scoreStdAcrossSeeds": round(float(np.std(seed_scores, axis=0).mean()), 4)},
        "baselineComparison": {
            "bothFlagged": int((is_anom & base_flags).sum()), "onlyIsolationForest": int((is_anom & ~base_flags).sum()),
            "onlyStatisticalBaseline": int((~is_anom & base_flags).sum()), "neitherFlagged": int((~is_anom & ~base_flags).sum()),
            "agreementPct": round(float(((is_anom == base_flags).mean()) * 100.0), 2), "scoreCorrelation": round(float(spearmanr(scores, base_max_z)[0]), 4)
        },
        "correlationMatrix": {"features": corr_feats, "matrix": [[round(float(v), 4) for v in row] for row in corr_df.values]},
        "records": records,
        "supervisedStatus": c["metadata"]["supervisedLabelAudit"],
        "evaluationNotes": [f"Trained in Google Colab with scikit-learn IsolationForest + SHAP TreeExplainer on {n:,} observations (no supervised metrics reported without ground truth)."]
    }

Path("outputs/astrocare_bundle.json").write_text(json.dumps(bundle_export))
print("Exported outputs/astrocare_bundle.json")

## 10. Download Trained `.joblib` Models & Web Bundle

In [ ]:
try:
    from google.colab import files
    for p in Path("models").glob("*.joblib"):
        files.download(str(p))
    if Path("outputs/astrocare_bundle.json").exists():
        files.download("outputs/astrocare_bundle.json")
except ImportError:
    print("Artifacts saved locally under models/ and outputs/astrocare_bundle.json.")